# Product and Category Performance Validation

**Owner:** Maheshwar  
**Assigned reviewer:** Mannan  
**Run after:** `03_gold_eda.ipynb`

Reconciles units, net revenue, and gross profit across sales-line, product, and category Gold tables.

This notebook is an owner-specific PySpark contribution. The owner must run it personally, inspect the displayed result, understand every assertion, and commit it from their own GitHub account.


## 1. Load the validated project tables


In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "analytics"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

products = spark.table(f"{CATALOG}.{SCHEMA}.products_silver")
sales_line = spark.table(f"{CATALOG}.{SCHEMA}.sales_line_gold")
product_performance = spark.table(f"{CATALOG}.{SCHEMA}.product_performance_gold")
category_performance = spark.table(f"{CATALOG}.{SCHEMA}.category_performance_gold")

print(f"products_silver: {products.count():,}")
print(f"sales_line_gold: {sales_line.count():,}")
print(f"product_performance_gold: {product_performance.count():,}")
print(f"category_performance_gold: {category_performance.count():,}")


## 2. Run owner-specific reconciliation and integrity checks


In [ ]:
# Product identifiers must remain unique in both the cleaned catalog and Gold summary.
assert products.select("ProductID").distinct().count() == products.count()
assert product_performance.select("ProductID").distinct().count() == product_performance.count()

# Every summarized product must exist in products_silver.
assert product_performance.join(
    products.select("ProductID"), "ProductID", "left_anti"
).count() == 0

line_totals = sales_line.agg(
    F.sum("Quantity").alias("Units"),
    F.round(F.sum("NetRevenue"), 2).alias("Revenue"),
    F.round(F.sum("GrossProfit"), 2).alias("Profit"),
).first()

product_totals = product_performance.agg(
    F.sum("UnitsSold").alias("Units"),
    F.round(F.sum("NetRevenue"), 2).alias("Revenue"),
    F.round(F.sum("GrossProfit"), 2).alias("Profit"),
).first()

category_totals = category_performance.agg(
    F.sum("UnitsSold").alias("Units"),
    F.round(F.sum("NetRevenue"), 2).alias("Revenue"),
    F.round(F.sum("GrossProfit"), 2).alias("Profit"),
).first()

assert line_totals["Units"] == product_totals["Units"] == category_totals["Units"]
assert abs(line_totals["Revenue"] - product_totals["Revenue"]) < 0.01
assert abs(line_totals["Revenue"] - category_totals["Revenue"]) < 0.01
assert abs(line_totals["Profit"] - product_totals["Profit"]) < 0.01
assert abs(line_totals["Profit"] - category_totals["Profit"]) < 0.01

assert product_performance.filter(F.col("UnitsSold") <= 0).count() == 0
assert category_performance.filter(F.col("NetRevenue") < 0).count() == 0


## 3. Display the observed business result and success marker


In [ ]:
top_category = category_performance.orderBy(F.desc("NetRevenue")).first()
top_product = product_performance.orderBy(F.desc("NetRevenue")).first()
print("Top category:", top_category.asDict())
print("Top product by revenue:", top_product.asDict())
display(category_performance.orderBy(F.desc("NetRevenue")))
display(
    product_performance.orderBy(F.desc("NetRevenue")).select(
        "ProductName", "Category", "UnitsSold", "NetRevenue", "GrossProfit"
    ).limit(10)
)

print("MAHESHWAR_PRODUCT_VALIDATION_PASSED")


## What the owner must be able to explain

- Which tables were compared and why.
- What each assertion protects against.
- What the displayed result means for FreshRoute.
- Why the final success marker `MAHESHWAR_PRODUCT_VALIDATION_PASSED` only prints after every check passes.
